# 06 Noise

The matrix imaging test from 05, with noise added to the reflection matrix R. Same forward model, imaging and
scoring as 05 (`mi_common.py`), so the only difference from 05 is the noise.

Each station pair gets noise with standard deviation `NSR x SIGNAL x sqrt(DAYS_REF / days)`, where `days` is how
long the pair recorded together. Pairs that share more time get cleaner correlations, so designs with short phases
or few overlapping stations suffer more. `SIGNAL` is the rms echo of the reference scenario recorded by the `grid`
design. The true noise level at Rotokawa is unknown, so a range of `NSR` is scanned.

Inputs: everything 05 reads. Output: `data/results_06_noise.csv`

In [ ]:
# --- Imports ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mi_common import MatrixImaging

In [ ]:
# --- Settings ---
PHASE = "p"
NSR_LIST = [0.25, 0.5, 1.0, 2.0]       # noise / signal for DAYS_REF days of shared recording. scanned
DAYS_REF = 30                          # recording time the NSR values refer to (days)
REF = "3d"                             # focusing model; "3d" isolates the effect of noise (07 covers model error)
SHAPES_RUN = ["dome", "lens"]          # fewer shapes keeps the run time down
DESIGNS = None                         # None = all, as in 05
RESULTS = "../data/results_06_noise.csv"

mi = MatrixImaging(PHASE, DAYS_REF=DAYS_REF)
RUN = mi.scenarios(SHAPES_RUN)
print(mi)
print(len(RUN), "scenarios x", len(NSR_LIST), "noise levels")

In [ ]:
# --- Reference signal: rms echo of a central dome, recorded by the grid design (noise-free) ---
ref_name = next((n for n in RUN if mi.SC["scenarios"][n]["centre"] == [0.0, 0.0]), RUN[0])
ids = sorted(set(mi.PERMANENT) | set(mi.D["designs"]["grid"][0]["stations"]))
k_sc, sig = mi.perturbation(ref_name)
R = mi.forward(ids, k_sc, sig)
M = mi.pair_mask(ids, mi.windows(mi.D["designs"]["grid"]))
SIGNAL = float(np.sqrt(np.mean(np.abs(R[:, M]) ** 2)))
print(f"reference: {ref_name}, rms signal {SIGNAL:.2e}")

In [ ]:
# --- Run: every noise level x scenario x design. Rerunning skips rows already in the results file ---
for nsr in NSR_LIST:
    noisy = mi.run(RUN, DESIGNS, results=RESULTS, extra=dict(nsr=nsr), noise=nsr * SIGNAL, ref=REF)
noisy = noisy[noisy.scenario.isin(RUN)]

In [ ]:
# --- Coverage vs noise: average and worst case over scenarios ---
print(noisy.groupby(["design", "nsr"]).coverage.agg(mean="mean", worst="min").round(2).unstack("nsr"))

# --- Phase 1: how far the picked BDT high was from the true one ---
print("\nPhase 1 location error (km)")
print(noisy[noisy.design == "phased (adaptive)"].groupby("nsr").p1_error_km.agg(median="median", worst="max").round(1))

In [ ]:
with plt.style.context("default"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.8))
    for dname, grp in noisy.groupby("design"):
        w = grp.groupby("nsr").coverage.min()
        ax1.plot(w.index, w.values, "o-", label=dname)
    ax1.set(xscale="log", xlabel=f"Noise / signal ({DAYS_REF} days shared)", ylabel="Worst-case coverage",
            ylim=(0, 1), title="Coverage")
    ax1.legend(fontsize=8)

    p = noisy[noisy.design == "phased (adaptive)"].groupby("nsr").p1_error_km
    ax2.plot(p.median().index, p.median().values, "o-", label="median")
    ax2.plot(p.max().index, p.max().values, "s--", label="worst")
    ax2.set(xscale="log", xlabel=f"Noise / signal ({DAYS_REF} days shared)", ylabel="Distance off (km)",
            title="Phase 1 location error")
    ax2.legend(fontsize=8)
    plt.tight_layout()
    plt.show()